# Text Embedding (Word2Vec)

The pipeline
```sh
viwik18
   ↓
Text cleaning
   ↓
Vietnamese word segmentation
   ↓
Tokenization
   ↓
Word2Vec
   ├── CBOW
   └── Skip-gram
   ↓
Word embeddings
   ↓
Evaluation
   ├── Similar words
   ├── Word similarity
   ├── Word analogy
   ├── Semantic arithmetic
   └── Visualization
```

## 0. Setup

In [4]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


## 1. Load Dataset

In [6]:
from pathlib import Path

DATA_DIR = Path("../data/viwik18-master/dataset")
OUTPUT_PATH = Path("../data/viwik18.txt")

files = sorted(DATA_DIR.glob("viwik18_*"))
files = files[:2]
with open(OUTPUT_PATH, "wb") as out_file:
    for file_path in files:
        with open(file_path, "rb") as in_file:
            out_file.write(in_file.read())

print("Merged:", OUTPUT_PATH)

Merged: ..\data\viwik18.txt


## 2. Text cleaning

In [7]:
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 3. Vietnamese word segmentation

In [8]:
from underthesea import word_tokenize

text = "internet society là một tổ chức quốc tế"
segmented = word_tokenize(text, format="text")
print(segmented)

internet society là một tổ_chức quốc_tế


## 4. Combine cleaning + segmentation + tokenization

In [9]:
def preprocess(text):
    text = clean_text(text)
    segmented = word_tokenize(text, format="text")
    tokens = segmented.split()
    return tokens

In [10]:
text = "Internet Society là một tổ chức quốc tế."

print(preprocess(text))

['internet', 'society', 'là', 'một', 'tổ_chức', 'quốc_tế']


In [11]:
class VietnameseWikiCorpus:
    def __init__(self, path, block_size=100_000):
        self.path = path
        self.block_size = block_size

    def __iter__(self):
        with open(self.path, "r", encoding="utf-8") as f:
            while True:
                text = f.read(self.block_size)
                if not text:
                    break
                text = clean_text(text)
                segmented = word_tokenize(text, format="text")

                tokens = segmented.split()

                if len(tokens) > 3:
                    yield tokens

## 5. Train a CBOW model

In [ ]:
import logging
from gensim.models import Word2Vec

logging.basicConfig(
    format="%(asctime)s : %(levelname)s : %(message)s",
    level=logging.INFO
)

corpus = VietnameseWikiCorpus("../data/viwik18.txt")

cbow_model = Word2Vec(
    sentences=corpus,
    vector_size=10,
    window=3,
    min_count=5,
    sg=0,
    negative=10,
    workers=2,
    epochs=2
)

## 6. Train Skip-gram

In [ ]:
skipgram_model = Word2Vec(
    sentences=corpus,
    vector_size=10,
    window=3,
    min_count=5,
    sg=1,              # Skip-gram
    negative=10,
    workers=2,
    epochs=5
)

## 7. Save models

In [ ]:
cbow_model.save("models/viwik18_cbow.model")
skipgram_model.save("models/vikil18_skipgram.model")

In [ ]:
from gensim.models import Word2Vec

cbow = Word2Vec.load("models/viwik18_cbow.model")

skipgram = Word2Vec.load("models/viwik18_skipgram.model")

## 8. Results

In [ ]:
vector = skipgram.wv["việt_nam"]

print(vector)
print(vector.shape)

In [ ]:
skipgram.wv.most_similar(
    "việt_nam",
    topn=10
)

In [ ]:
test_words = [
    "việt_nam",
    "hà_nội",
    "máy_tính",
    "bóng_đá",
    "sinh_viên",
    "giáo_viên"
]

for word in test_words:

    if word not in skipgram.wv:
        continue

    print(f"\n{word}")

    for w, score in skipgram.wv.most_similar(
        word,
        topn=5
    ):
        print(f"{w:20} {score:.4f}")

In [ ]:
# Word Similarity
correlation_1 = skipgram.wv.similarity(
    "việt_nam",
    "hà_nội"
)
correlation_2 = skipgram.wv.similarity(
    "việt_nam",
    "máy_tính"
)
print("First correlation: ", correlation_1)
print("Second correlation: ", correlation_2)

In [ ]:
import numpy as np
# Implement cosine similarity 
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2)
    )

v1 = skipgram.wv["việt_nam"]
v2 = skipgram.wv["hà_nội"]

similarity = cosine_similarity(v1, v2)
print(similarity)



## 9. Visualize 

### 1. Reduce dimensions to 2D using t-SNE

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
words = [
    "việt_nam",
    "hà_nội",
    "trung_quốc",
    "nhật_bản",
    "pháp",
    "đức",
    "máy_tính",
    "công_nghệ",
    "internet",
    "bóng_đá",
    "thể_thao"
]

words = [
    word for word in words
    if word in skipgram.wv
]

In [ ]:
X = np.array([
    skipgram_model.wv[word]
    for word in words
])

print(X.shape)
vector_size=100

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=2)
y = tsne.fit_transform(X)

# 5. Plot the embeddings

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y[:, 0], y[:, 1])
for i, word in enumerate(words):
    plt.annotate(word, xy=(y[i, 0], y[i, 1]), xytext=(5, 2), textcoords='offset points')
plt.title("t-SNE Visualization of Word2Vec Embeddings")
plt.show()

# Reference
- [Medium - Train a Word2Vec model from scratch with gensim](https://medium.com/data-science/how-to-train-a-word2vec-model-from-scratch-with-gensim-c457d587e031)
- [Github - Build and visualize Word2Vec model with Gensim](https://github.com/MiguelSteph/word2vec-with-gensim)